In [386]:
import numpy as np
from abc import ABC, abstractmethod

from alex_area.movie_generator.buffer_movies import BufferMovie, load_buffer_movies

In [95]:
MIN_FRAME_DIM_SIZE: int = 42

In [96]:
B_MOV_X_MAX: int = 231
B_MOV_Y_MAX: int = 163

# load buffer movies and crop
b_movs = load_buffer_movies()[12::]  # 12 onwards for TwoMP large FoV buffer movies
BUFFER_MOVIES = [
    mov[:, :B_MOV_Y_MAX, :B_MOV_X_MAX] for mov in b_movs
]  # crop to same size

In [153]:
def gen_random_mov_stack() -> BufferMovie:

    rand_index = np.random.randint(0, len(BUFFER_MOVIES))

    mov: BufferMovie = BUFFER_MOVIES[rand_index]

    x_width = np.random.randint(MIN_FRAME_DIM_SIZE, B_MOV_X_MAX + 1)
    y_width = np.random.randint(MIN_FRAME_DIM_SIZE, B_MOV_Y_MAX + 1)

    x_max = B_MOV_X_MAX - x_width
    y_max = B_MOV_Y_MAX - y_width

    y_rand = np.random.randint(0, y_max + 1)
    x_rand = np.random.randint(0, x_max + 1)

    return mov[:, y_rand : y_rand + y_width, x_rand : x_rand + x_width]

In [384]:
rand_mov = np.random.randint(
    low=np.iinfo(np.uint16).min,
    high=np.iinfo(np.uint16).max + 1,
    size=(2500, MIN_FRAME_DIM_SIZE, MIN_FRAME_DIM_SIZE),
    dtype=np.uint16,
)

In [ ]:
class AbstractSimEvent(ABC):

    @property
    @abstractmethod
    def x(self) -> float:
        pass

    @property
    @abstractmethod
    def y(self) -> float:
        pass

    @property
    @abstractmethod
    def i(self) -> float:
        pass

    @property
    @abstractmethod
    def c(self) -> float:
        pass

    @property
    @abstractmethod
    def hot_px(self) -> tuple[int, int]:
        pass

    @property
    @abstractmethod
    def offset(self) -> tuple[float, float]:
        pass

    @abstractmethod
    def to_simple(self) -> list[list[float]]:
        pass

In [ ]:
class BaseSimEvent(AbstractSimEvent):

    def __init__(
        self,
        x: float,
        y: float,
        i: float,
        c: float,
    ) -> None:

        self.x_pos = x
        self.y_pos = y
        self.i_time = i
        self.contrast = c

        return

    @property
    def x(self) -> float:
        return self.x_pos

    @property
    def y(self) -> float:
        return self.y_pos

    @property
    def i(self) -> float:
        return self.i_time

    @property
    def c(self) -> float:
        return self.contrast

    @property
    def hot_px(self) -> tuple[int, int]:
        return (np.round(self.x).astype(int), np.round(self.y).astype(int))

    @property
    def offset(self) -> tuple[float, float]:

        x_px, y_px = self.hot_px

        return (self.x - float(x_px), self.y - float(y_px))

    def to_simple(self) -> list[list[float]]:
        return [[self.x, self.y, self.i, self.c]]

In [ ]:
class BindingSimEvent(BaseSimEvent):

    def __init__(self, x: float, y: float, i: float, c: float) -> None:
        super().__init__(x, y, i, c)


class UnbindingSimEvent(BaseSimEvent):

    def __init__(self, x: float, y: float, i: float, c: float) -> None:
        super().__init__(x, y, i, c)


class MovementSimEvent(BaseSimEvent):

    def __init__(
        self, x_mid: float, y_mid: float, i: float, c: float, r: float, theta: float
    ) -> None:
        super().__init__(x_mid, y_mid, i, c)

        self.distance: float = r
        self.theta: float = (theta / (2 * np.pi)) - np.floor(theta / (2 * np.pi))

        self.dx, self.dy = (r * np.cos(theta), r * np.sin(theta))

        self.unbinding = UnbindingSimEvent(
            x=x_mid - self.dx / 2, y=y_mid - self.dy / 2, i=i, c=c
        )
        self.binding = BindingSimEvent(
            x=x_mid + self.dx / 2, y=y_mid + self.dy / 2, i=i, c=c
        )

    def to_simple(self) -> list[list[float]]:

        return self.unbinding.to_simple() + self.binding.to_simple()

In [ ]:
def gen_events(
    n_binding: int,
    n_unbinding: int,
    n_movement: int,
): 
    pass